In [1]:
import warnings
import math
import numpy as np
from numpy.exceptions import ComplexWarning

# GPAW triggers harmless RuntimeWarnings (divide by zero, overflow in dot/matmul)
# on Apple Silicon due to numerical edge cases in the PAW setup and LFC phases.
# These do not affect results.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="gpaw")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="numpy")
warnings.filterwarnings("ignore", category=ComplexWarning)

import sys
from pathlib import Path

repo_root = Path.cwd().parent   # .../Bloch-PAW-main
sys.path.insert(0, str(repo_root))

# Metallic hydrogen

In [2]:
from ase.build import bulk
from gpaw import GPAW

from bloch_paw.extractor import PawExtractor
from bloch_paw import PawReader, OneNormCalculator, ResourceEstimator

results = []

atoms = bulk('H', 'fcc', a=3.67)  # build once; GPAW will overwrite calc each loop
Nb = 5

data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

for i in range(1, 4):
    k_mesh = (i, i, i)
    k_tag = f"{i}x{i}x{i}"  # nicer filename

    # Fresh calculator each iteration
    calc = GPAW(
        mode="lcao", basis="dzp", h=0.21,
        kpts={"size": k_mesh, "gamma": True},
        xc="PBE", nbands=Nb,
        symmetry={"point_group": False, "time_reversal": False},
        txt=str(data_dir / f"gpaw_{k_tag}.txt"),  # optional: per-run log
    )
    atoms.calc = calc

    # Run SCF
    energy = atoms.get_potential_energy()

    # Export Bloch-PAW ingredients
    extractor = PawExtractor(
        calc, nbands=Nb,
        thr_rho=1.0e-16, thr_D=1.0e-19, thr_C=1.0e-7, thr_h=1e-5, thr_kappa=1e-5,
    )

    h5_path = data_dir / f"lcbo_{k_tag}.h5"
    extractor.export_hdf5(filepath=str(h5_path), write_two_body=False)

    # Read + resource estimation
    reader = PawReader(str(h5_path))
    with reader:
        inputs = reader.to_calculator_inputs(lazy=True)

        one_norm_calc = OneNormCalculator(
            **inputs, thr_rank=3e-5, sv_floor=1e-12, scale_floor=1e-12
        )
        lam = one_norm_calc.lambda_one_norm()
        R_avg, R0 = one_norm_calc.compute_average_rank()

        eps_chem = 27e-3
        eps_qpe = eps_chem / 5

        est = ResourceEstimator.from_hdf5(str(h5_path))
        toffolis = est.toffoli_count_per_be(Rl=R_avg, R0=R0)
        qubits = est.total_qubits(Rl=R_avg, R0=R0, lam=lam, eps_qpe=eps_qpe)
        iters = math.ceil(math.pi * lam / eps_qpe)

    total_toffolis = toffolis * iters

    print("=" * 60)
    print(f"Resource estimates for k mesh = {k_mesh}")
    print("=" * 60)
    print(f"SCF energy (eV) = {energy:.6f}")
    print(f"One-norm λ = {lam:.6f}")
    print(f"Toffoli per query:         {toffolis:,}")
    print(f"QPE iterations:            {iters:,}")
    print(f"Total Toffoli:             {total_toffolis:,}")
    print(f"Logical qubits:            {qubits:,}")
    print(f"HDF5: {h5_path.resolve()}")
    print("=" * 60)

    results.append({
        "k_mesh": k_mesh,
        "lambda": lam,
        "toffoli_per_query": toffolis,
        "total_toffoli": total_toffolis,
        "logical_qubits": qubits 
    })

results

Resource estimates for k mesh = (1, 1, 1)
SCF energy (eV) = -4.334878
One-norm λ = 109.510551
Toffoli per query:         7,377
QPE iterations:            63,711
Total Toffoli:             469,996,047
Logical qubits:            1,114
HDF5: /Users/rbhardwaj/Desktop/Fe Simulation/Code/Bloch-PAW-main/data/lcbo_1x1x1.h5
Resource estimates for k mesh = (2, 2, 2)
SCF energy (eV) = -0.581486
One-norm λ = 768.994889
Toffoli per query:         43,488
QPE iterations:            447,384
Total Toffoli:             19,455,835,392
Logical qubits:            7,037
HDF5: /Users/rbhardwaj/Desktop/Fe Simulation/Code/Bloch-PAW-main/data/lcbo_2x2x2.h5
Resource estimates for k mesh = (3, 3, 3)
SCF energy (eV) = -1.134186
One-norm λ = 3047.926301
Toffoli per query:         132,843
QPE iterations:            1,773,212
Total Toffoli:             235,558,801,716
Logical qubits:            21,642
HDF5: /Users/rbhardwaj/Desktop/Fe Simulation/Code/Bloch-PAW-main/data/lcbo_3x3x3.h5


[{'k_mesh': (1, 1, 1),
  'lambda': 109.51055116966349,
  'toffoli_per_query': 7377,
  'total_toffoli': 469996047,
  'logical_qubits': 1114},
 {'k_mesh': (2, 2, 2),
  'lambda': 768.994888696316,
  'toffoli_per_query': 43488,
  'total_toffoli': 19455835392,
  'logical_qubits': 7037},
 {'k_mesh': (3, 3, 3),
  'lambda': 3047.926300767696,
  'toffoli_per_query': 132843,
  'total_toffoli': 235558801716,
  'logical_qubits': 21642}]